# Tydzień 12: Wdrożenie modelu – aplikacja z prawdziwymi predykcjami i publikacja w internecie 🚀

| | |
|---|---|
| **Rozdział książki** | 8. *Deploying a Machine Learning Model* – podrozdziały: *Developing the Web Application*, *Running the Web Application Locally*, *The Iris Classification App*, *Deploying an Application to Streamlit Cloud* + 9. *Closing Thoughts* |
| **Czas zajęć** | 90 minut |
| **Dane / model** | model `regression_model.pkl` zapisany w tygodniu 4 (jeśli go nie masz – notebook wytrenuje go automatycznie) |
| **Nowe narzędzia** | `load_model()` w aplikacji, `@st.cache_resource`, `requirements.txt`, Streamlit Community Cloud |

## 🎯 Cel zajęć
Po dzisiejszych zajęciach będziesz:
1. umieć **połączyć zapisany model PyCaret z aplikacją Streamlit** i przewidywać koszty „na żywo”,
2. rozumieć, po co jest **pamięć podręczna** (*cache*) i plik **`requirements.txt`**,
3. umieć **automatycznie przetestować** aplikację,
4. wiedzieć, jak **opublikować** aplikację w internecie w **Streamlit Community Cloud**,
5. mieć w głowie **mapę całego kursu** – od danych do działającej aplikacji.

## 🗓️ Plan zajęć
| Czas | Co robimy |
|---|---|
| 0–10 min | 📚 Teoria: od pliku `.pkl` do aplikacji w internecie |
| 10–55 min | 👣 Krok po kroku: model + aplikacja + test |
| 55–70 min | ☁️ Publikacja w Streamlit Community Cloud (pokaz / samodzielnie) |
| 70–85 min | 🧑‍💻 Ćwiczenie: porównanie ze średnią – ✅ **obowiązkowe** (zaliczenie) |
| 85–90 min | 🎓 Podsumowanie całego kursu |
| poza zajęciami | 🏋️ Część 4: ćwiczenia dodatkowe (trudniejsze, nieobowiązkowe) – dla szybszych osób lub do domu |

---
# 📚 Część 1: Wprowadzenie teoretyczne

## Od słoika w zamrażarce do restauracji 🫙➡️🍽️
W tygodniu 4 „zamroziliśmy” model w pliku `regression_model.pkl`. Ten plik zawiera **cały potok** (*pipeline*):
kodowanie one-hot, transformację, normalizację **i** wytrenowany model Gradient Boosting. Dzięki temu aplikacja może podać
modelowi surowe dane z formularza (`'male'`, `'yes'`, `'southwest'`…), a potok sam je przygotuje – dokładnie tak jak podczas treningu.

Aplikacja = **restauracja**: klient (użytkownik) składa zamówienie (formularz), kuchnia (model) przygotowuje danie (predykcję),
kelner (Streamlit) podaje wynik na talerzu (stronie internetowej).

## Pamięć podręczna – `@st.cache_resource` ♨️
Pamiętasz, że Streamlit uruchamia **cały skrypt od nowa** po każdym kliknięciu? Wczytywanie modelu z pliku przy każdym kliknięciu
byłoby jak **gotowanie wody na herbatę od zera dla każdego gościa**. Dekorator `@st.cache_resource` wczytuje model **raz**
i trzyma go „na gorąco” w pamięci.

⚠️ W książce użyto `@st.cache(allow_output_mutation=True)`. **Ta funkcja została usunięta** z nowych wersji Streamlit –
jej następcą jest właśnie `@st.cache_resource` (do modeli, połączeń z bazą) i `@st.cache_data` (do tabel z danymi).

## Plik `requirements.txt` 📜
To **lista zakupów** dla serwera: jakie biblioteki (i w jakich wersjach) trzeba zainstalować, żeby aplikacja działała.
Serwer w chmurze nie wie, co mamy zainstalowane w Codespaces – musimy mu to napisać.

## Streamlit Community Cloud ☁️
Darmowa usługa firmy Streamlit (Snowflake), która publikuje aplikacje **prosto z repozytorium GitHub**:
podajesz repozytorium i plik `app.py`, klikasz **Deploy** – i po kilku minutach aplikacja ma publiczny adres, który możesz wysłać
znajomym. Gdy zmienisz kod na GitHubie, aplikacja sama się zaktualizuje.

---
# 👣 Część 2: Krok po kroku

### Krok 1: Czy mamy model z tygodnia 4? 🔎

In [ ]:
# Wyłączamy ostrzeżenia (warnings), żeby nie zaśmiecały ekranu.
# Ostrzeżenie to NIE błąd – to tylko podpowiedź dla programistów bibliotek.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# os – operacje na plikach i folderach; shutil – kopiowanie plików; pandas – tabele
import os
import shutil
import pandas as pd

In [ ]:
# Ścieżka do modelu zapisanego w tygodniu 4. os.path.exists zwraca True, jeśli plik istnieje
model_path = '../modele/regression_model.pkl'
os.path.exists(model_path)

In [ ]:
# 🆘 Plan awaryjny: jeśli powyżej wyszło False, ta komórka szybko wytrenuje i zapisze model (ok. 30 s).
# Jeśli wyszło True – komórka nic nie zmieni, tylko potwierdzi, że model jest gotowy.
if not os.path.exists(model_path):
    from pycaret.datasets import get_data
    from pycaret.regression import setup, create_model, finalize_model, save_model
    data = get_data('insurance', verbose=False)
    setup(data=data, target='charges', session_id=7402, verbose=False)
    os.makedirs('../modele', exist_ok=True)
    save_model(finalize_model(create_model('gbr', verbose=False)), '../modele/regression_model', verbose=False)
print("Model gotowy:", os.path.exists(model_path))

### Krok 2: Kopiujemy model do folderu aplikacji 📦
Aplikacja (i później serwer w chmurze) powinna mieć **wszystko w jednym folderze**: kod, model i listę bibliotek.

In [ ]:
# Tworzymy folder aplikacji (gdyby nie istniał) i kopiujemy do niego model
os.makedirs('../aplikacja', exist_ok=True)
shutil.copy(model_path, '../aplikacja/regression_model.pkl')

### Krok 3: Próba generalna w notebooku 🧪
Zanim napiszemy aplikację, sprawdźmy w notebooku, że wczytanie modelu i predykcja działają.
Użyjemy osoby z przykładu w książce: 35 lat, mężczyzna, BMI 20, bez dzieci, region *southwest*, **pali**.

In [ ]:
# Z PyCaret potrzebujemy tylko dwóch funkcji: wczytania modelu i predykcji
from pycaret.regression import load_model, predict_model

In [ ]:
# Wczytujemy model z folderu aplikacji (rozszerzenie .pkl dopisze się samo)
model = load_model('../aplikacja/regression_model')

In [ ]:
# Jeden wiersz danych – dokładnie w formacie, jaki zbuduje formularz z tygodnia 11
person = pd.DataFrame([{'age': 35, 'sex': 'male', 'bmi': 20.0, 'children': 0,
                        'smoker': 'yes', 'region': 'southwest'}])

In [ ]:
# Predykcja: interesuje nas pierwsza (i jedyna) wartość w kolumnie prediction_label
predictions = predict_model(model, data=person)
predictions['prediction_label'][0]

🔎 Model przewiduje ok. **17–18 tys. $** rocznie (w książce: 17 069 $; drobna różnica wynika z innej wersji PyCaret
i innych ustawień strojenia). Działa! Teraz przeniesiemy ten kod do aplikacji.

### Krok 4: Aplikacja z modelem – kawałek po kawałku 🧱
Budujemy plik `app.py` od nowa (zastąpi wersję z zeszłego tygodnia).

#### 4a. Importy i wczytanie modelu (z pamięcią podręczną)

In [ ]:
%%writefile ../aplikacja/app.py
# ============================================================
#  Kalkulator kosztów ubezpieczenia – wersja z modelem ML
#  Uruchomienie (z głównego folderu repozytorium):  streamlit run aplikacja/app.py
# ============================================================
from pathlib import Path

import pandas as pd
import streamlit as st
from pycaret.regression import load_model, predict_model

# Ustawienia strony – musi być pierwszym poleceniem st
st.set_page_config(page_title="Kalkulator kosztów ubezpieczenia")

# Folder, w którym leży ten plik – model znajdziemy niezależnie od tego, skąd uruchomimy aplikację
APP_DIR = Path(__file__).parent


@st.cache_resource  # wczytaj model tylko RAZ i trzymaj go w pamięci podręcznej (cache)
def get_model():
    return load_model(str(APP_DIR / 'regression_model'))


def predict(model, df):
    """Zwraca przewidywany koszt dla pierwszego wiersza tabeli df."""
    predictions = predict_model(model, data=df)
    return predictions['prediction_label'][0]


model = get_model()

#### 4b. Nagłówek i opis

In [ ]:
%%writefile -a ../aplikacja/app.py

st.title("💰 Kalkulator kosztów ubezpieczenia")
st.markdown("Wpisz swoje dane, a model uczenia maszynowego (Gradient Boosting, PyCaret) przewidzi "
            "**roczny koszt ubezpieczenia zdrowotnego**. Aplikacja z kursu *Praktyczny Machine Learning z PyCaret*.")

#### 4c. Formularz (te same widżety co w tygodniu 11)

In [ ]:
%%writefile -a ../aplikacja/app.py

# --- Formularz ---
form = st.form("charges")
age = form.number_input("Wiek", min_value=18, max_value=64, value=30)
sex = form.radio("Płeć", ["Mężczyzna", "Kobieta"])
bmi = form.number_input("BMI (wskaźnik masy ciała)", min_value=15.0, max_value=55.0, value=25.0, step=0.1)
children = form.slider("Liczba dzieci", min_value=0, max_value=5, value=0)
region = form.selectbox("Region USA", ["southwest", "southeast", "northwest", "northeast"])
smoker = form.checkbox("Palę papierosy")
submitted = form.form_submit_button("Oblicz koszt")

#### 4d. Tłumaczenie danych i predykcja „na żywo” 🔮

In [ ]:
%%writefile -a ../aplikacja/app.py

# --- Dane z formularza -> jeden wiersz w formacie danych treningowych ---
input_df = pd.DataFrame([{
    'age': age,
    'sex': 'male' if sex == 'Mężczyzna' else 'female',
    'bmi': bmi,
    'children': children,
    'smoker': 'yes' if smoker else 'no',
    'region': region,
}])

# --- Po kliknięciu przycisku: predykcja modelem i wynik na stronie ---
if submitted:
    out = predict(model, input_df)
    st.success(f"Przewidywany roczny koszt ubezpieczenia: **{out:,.2f} $**")

In [ ]:
# Podgląd całego pliku – sprawdź, czy nic się nie zdublowało
print(open('../aplikacja/app.py', encoding='utf-8').read())

### Krok 5: Lista bibliotek dla serwera – `requirements.txt` 📜
Do działania aplikacji wystarczy „lekka” wersja PyCaret (bez dodatków `[full]`) i Streamlit.
Wersje przypinamy (`==`), żeby serwer zainstalował **dokładnie te same** biblioteki, z którymi zapisaliśmy model.

In [ ]:
%%writefile ../aplikacja/requirements.txt
pycaret==3.3.2
scikit-learn==1.4.2
streamlit==1.65.0

### Krok 6: Uruchamiamy aplikację lokalnie 🖥️
W terminalu (z głównego folderu repozytorium):
```bash
streamlit run aplikacja/app.py
```
Pierwsze uruchomienie potrwa kilka sekund dłużej (import PyCaret + wczytanie modelu). Wypełnij formularz i kliknij **„Oblicz koszt”**.
Spróbuj zaznaczyć i odznaczyć „Palę papierosy” – jak bardzo zmienia się koszt? 🚬💸
Zatrzymanie: <kbd>Ctrl</kbd> + <kbd>C</kbd> w terminalu.

### Krok 7: Automatyczny test aplikacji – bez przeglądarki 🤖
Programiści nie klikają ręcznie po każdej zmianie – piszą **testy automatyczne**. Streamlit ma do tego narzędzie `AppTest`,
które uruchamia aplikację „na niby” i pozwala „klikać” z poziomu kodu.

💡 Komunikat *„missing ScriptRunContext! This warning can be ignored…”* możesz spokojnie zignorować – sam Streamlit pisze, że jest niegroźny.

In [ ]:
# AppTest uruchamia skrypt aplikacji bez przeglądarki; potrzebuje PEŁNEJ ścieżki do pliku (os.path.abspath)
from streamlit.testing.v1 import AppTest
app_file = os.path.abspath('../aplikacja/app.py')
at = AppTest.from_file(app_file, default_timeout=120).run()   # default_timeout – ile sekund czekać na wynik

In [ ]:
# "Wypełniamy" formularz: wiek 35, BMI 20, zaznaczamy palenie i klikamy przycisk formularza
at.number_input[0].set_value(35)
at.number_input[1].set_value(20.0)
at.checkbox[0].check();   # średnik ukrywa techniczny opis widżetu

In [ ]:
# Kliknięcie przycisku i ponowne uruchomienie skryptu (tak jak po prawdziwym kliknięciu)
at.button[0].click()
at.run();   # średnik ukrywa długi techniczny opis testu

In [ ]:
# Co wyświetliła aplikacja? Lista zielonych komunikatów success
[s.value for s in at.success]

🔎 Jeśli widzisz komunikat z przewidywanym kosztem (ok. 17–18 tys. $) – aplikacja działa poprawnie! ✅
(Region domyślny to *southwest*, płeć *Mężczyzna*, liczba dzieci 0 – czyli dokładnie osoba z kroku 3.)

---
## ☁️ Krok 8: Publikacja w Streamlit Community Cloud

Ten krok wykonujemy poza notebookiem. Potrzebne jest **konto GitHub** (to samo, którego używasz w Codespaces).

### 8a. Kod musi trafić do Twojego repozytorium na GitHubie
1. W VS Code otwórz panel **Source Control** (ikona rozgałęzienia po lewej, skrót <kbd>Ctrl</kbd> + <kbd>Shift</kbd> + <kbd>G</kbd>).
2. Przy folderze `aplikacja/` (pliki `app.py`, `regression_model.pkl`, `requirements.txt`) kliknij **+** (*Stage*).
3. Wpisz opis zmian, np. `Aplikacja: kalkulator kosztów ubezpieczenia`, i kliknij **Commit**, a potem **Sync Changes** / **Push**.
4. 💡 Jeśli pracujesz na repozytorium prowadzącego (bez prawa zapisu), Codespaces zaproponuje **utworzenie forka**
   (własnej kopii repozytorium) – zgódź się.

### 8b. Wdrożenie
1. Wejdź na **[share.streamlit.io](https://share.streamlit.io)** i zaloguj się przez **GitHub** (*Continue with GitHub*).
2. Kliknij **Create app** → **Deploy a public app from GitHub**.
3. Wypełnij pola:
   - **Repository:** `twoja-nazwa/ML_WDIB` (Twój fork),
   - **Branch:** `main`,
   - **Main file path:** `aplikacja/app.py`,
   - **App URL:** wymyśl własny adres, np. `kalkulator-ubezpieczen-ania`.
4. Kliknij **Advanced settings** i wybierz **Python 3.11** (lub 3.10). ⚠️ **To ważne:** PyCaret 3 nie działa na Pythonie 3.12+!
5. Kliknij **Deploy** i poczekaj ok. **5–10 minut** (serwer instaluje PyCaret – to spora biblioteka).
6. Gotowe! 🎉 Aplikacja ma publiczny adres `https://twoj-adres.streamlit.app` – wyślij go znajomym.

### 8c. Gdy coś nie działa 🔧
- W prawym dolnym rogu opublikowanej aplikacji kliknij **Manage app** – zobaczysz **logi** serwera (komunikaty błędów).
- `ModuleNotFoundError` → brakuje biblioteki w `aplikacja/requirements.txt`.
- `FileNotFoundError: regression_model.pkl` → model nie został wysłany na GitHuba (sprawdź krok 8a).
- Błędy instalacji `pycaret` → sprawdź, czy w *Advanced settings* wybrano Pythona 3.10 lub 3.11 (zmiana wersji wymaga usunięcia i ponownego wdrożenia aplikacji).

---
# 🧑‍💻 Część 3: Ćwiczenie dla Ciebie
> ✅ **Ćwiczenie obowiązkowe** – na jego podstawie zaliczasz dzisiejsze zajęcia (zaliczone / niezaliczone).

**Zadanie: Jak wypadam na tle innych?** 📊

Średni roczny koszt ubezpieczenia w danych treningowych to ok. **13 270 $** (pamiętasz `data.describe()` z tygodnia 1?).

- **a)** Dopisz do aplikacji fragment, który **po obliczeniu kosztu** pokaże:
  - żółte ostrzeżenie (`st.warning`), jeśli przewidywany koszt jest **wyższy** od średniej,
  - niebieską informację (`st.info`), jeśli jest **niższy lub równy** średniej.
- **b)** Sprawdź działanie aplikacji: w przeglądarce **albo** testem `AppTest` (kroki 7).

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- Dopisz kod komórką `%%writefile -a ../aplikacja/app.py`.
- Zmienna `out` istnieje tylko wtedy, gdy kliknięto przycisk – więc cały nowy kod musi być **wewnątrz** `if submitted:`.
- Różnica w procentach: `(out - average) / average * 100`.
- Formatowanie liczby bez miejsc po przecinku: `f"{liczba:.0f}"`.

</details>

In [ ]:
# ✍️ Twój kod tutaj (możesz dodawać kolejne komórki przyciskiem "+ Code")

### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
## 🏠 Zadanie domowe (dla chętnych): aplikacja do rozpoznawania irysów 🌸
W rozdziale 8 książki jest druga aplikacja – **klasyfikator irysów** oparty na modelu LDA z tygodnia 6
(`../modele/classification_model.pkl`). Zbuduj ją samodzielnie w folderze `aplikacja_iris/`:
4 suwaki (`sepal_length`, `sepal_width`, `petal_length`, `petal_width`, zakres 0–10 cm, krok 0,1) i przycisk „Rozpoznaj gatunek”.
Uruchomienie: `streamlit run aplikacja_iris/app.py`.

---
# 🏋️ Część 4: Ćwiczenia dodatkowe – trochę trudniejsze
Dla osób, które skończyły ćwiczenie podstawowe, albo **do samodzielnej pracy w domu**.
Ćwiczenia z tej części są **nieobowiązkowe** – nie są wymagane do zaliczenia zajęć.

Te zadania mają mniej podpowiedzi w treści, łączą kilka poznanych narzędzi i czasem wprowadzają **jedną nową funkcję**
(zawsze opisaną w podpowiedzi). Poziomy trudności: ⭐⭐ – średnie, ⭐⭐⭐ – wyzwanie.

## ⭐⭐ Ćwiczenie 2: „Co by było, gdyby…?” – symulacja rzucenia palenia
Dodaj do aplikacji **symulację**: po obliczeniu kosztu pokaż też koszt **tej samej osoby** z przeciwną odpowiedzią na pytanie
o palenie, jako „kafelek” `st.metric` z **różnicą** (`delta`). Przykład: „Gdybyś nie palił(a): 4 900 $ (−12 700 $)”.

- **a)** Dopisz kod do `app.py` (wewnątrz `if submitted:`).
- **b)** Sprawdź działanie testem `AppTest` dla osoby **palącej**: jaka jest różnica kosztów?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- Kopia tabeli z danymi z formularza: `other_df = input_df.copy()`, a potem zmiana jednej kolumny:
  `other_df['smoker'] = 'no' if smoker else 'yes'`.
- Drugi koszt policzysz tą samą funkcją: `predict(model, other_df)`.
- `st.metric(etykieta, wartość, delta=różnica, delta_color="inverse")` – `delta` to zmiana pokazana strzałką;
  `"inverse"` sprawia, że **spadek** kosztu jest zielony (dobrze), a wzrost – czerwony.
- W `AppTest` kafelki są w liście `at.metric`; wartość: `.value`, różnica: `.delta`.

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

## ⭐⭐⭐ Ćwiczenie 3: Wykres – jak zmieniałby się mój koszt z wiekiem?
Pokaż w aplikacji **wykres liniowy**: przewidywany koszt dla użytkownika w każdym wieku od 18 do 64 lat
(wszystkie inne cechy bez zmian). Zrób to **jednym** wywołaniem modelu dla 47 wierszy naraz (tzw. predykcja wsadowa, *batch prediction*),
a nie 47 osobnymi wywołaniami.

- **a)** Dopisz do `app.py` (wewnątrz `if submitted:`) kod, który buduje tabelę 47 „wersji” użytkownika różniących się tylko wiekiem,
  przewiduje dla nich koszty i rysuje wykres `st.line_chart`.
- **b)** Sprawdź w `AppTest`, że aplikacja działa bez błędów (`at.exception` jest puste) i że pojawił się nowy podtytuł.
- **c)** Uruchom aplikację w przeglądarce i porównaj kształt krzywej dla osoby palącej i niepalącej.

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- 47 kopii jednego wiersza: `pd.concat([input_df] * 47, ignore_index=True)`.
- Nadpisanie kolumny wiekiem 18–64: `curve_df['age'] = list(range(18, 65))`.
- Predykcja dla całej tabeli: `predict_model(model, data=curve_df)` – kolumna `prediction_label` ma 47 wartości.
- Wykres: `st.line_chart(tabela, x='age', y='prediction_label')`.
- W `AppTest` podtytuły są w `at.subheader`.

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
# 🎓 Podsumowanie całego kursu

## Co potrafisz po 12 tygodniach? 💪
| Tydzień | Temat | Najważniejsze umiejętności |
|---|---|---|
| 1 | Wstęp do ML | rodzaje ML, cechy i zmienna celu, `get_data()` |
| 2 | Regresja – EDA | histogramy, wykresy rozrzutu, korelacja |
| 3 | Regresja – `setup()` | podział train/test, normalizacja, one-hot, walidacja krzyżowa, `compare_models()` |
| 4 | Regresja – strojenie | `create_model()`, `tune_model()`, `predict_model()`, `save_model()` |
| 5 | Klasyfikacja – EDA | balans klas, wykresy pudełkowe, kodowanie etykiet |
| 6 | Klasyfikacja – ocena | dokładność, macierz pomyłek, granice decyzyjne |
| 7 | Grupowanie | K-Means, metoda łokcia, sylwetka, PCA |
| 8 | Anomalie | LOF, Isolation Forest, UMAP |
| 9 | NLP | worek słów, chmury słów, LDA – modelowanie tematów |
| 10 | Szeregi czasowe | trend, sezonowość, wygładzanie wykładnicze, prognozy |
| 11 | Streamlit cz. 1 | widżety i formularze |
| 12 | Wdrożenie | model w aplikacji, testy, publikacja w chmurze |

## 🧾 Ściągawka PyCaret – „przepis”, który działa w każdym module
```python
from pycaret.regression import *        # albo: classification, clustering, anomaly, time_series

exp = setup(data, target='kolumna', session_id=123)   # 1. przygotuj dane
best = compare_models()                               # 2. porównaj modele
model = create_model('id_modelu')                     # 3. zbuduj wybrany model
tuned = tune_model(model)                             # 4. dostrój hiperparametry
plot_model(tuned, plot='...')                         # 5. narysuj wykresy
predict_model(tuned)                                  # 6. sprawdź na zbiorze testowym
final = finalize_model(tuned)                         # 7. trenuj na wszystkich danych
save_model(final, 'nazwa')                            # 8. zapisz do pliku .pkl
model = load_model('nazwa')                           # 9. wczytaj i używaj (np. w aplikacji)
```
(W grupowaniu i anomaliach zamiast `predict_model` zwykle używamy `assign_model`, a w szeregach czasowych podajemy `fh=` zamiast `target=`.)

## 🚀 Co dalej?
- 📖 **Dokumentacja PyCaret:** https://pycaret.readthedocs.io – wiele funkcji, których nie zdążyliśmy poznać (np. `interpret_model`, `ensemble_model`, `blend_models`).
- 📘 **Teoria:** bezpłatna książka *An Introduction to Statistical Learning* (https://www.statlearning.com).
- 🏆 **Praktyka:** konkursy i zbiory danych na **Kaggle** (https://www.kaggle.com) – zacznij od „Titanic”.
- 🧰 **Zajrzyj „pod maskę”:** PyCaret korzysta ze **scikit-learn** – warto poznać tę bibliotekę bezpośrednio.
- 💼 **Portfolio:** wrzuć swoje notebooki i aplikację na GitHub – to świetna wizytówka dla pracodawcy!

> *„Edukacja to proces, który nigdy się nie kończy”* – parafrazując zakończenie książki Gianisa Toliosa.
> Dziękujemy za wspólny semestr i powodzenia w dalszej przygodzie z uczeniem maszynowym! 🎉

## 📖 Słowniczek PL / EN
| Polski | English | Znaczenie |
|---|---|---|
| wdrożenie | deployment | udostępnienie modelu użytkownikom |
| pamięć podręczna | cache | przechowywanie wyniku, by nie liczyć go ponownie |
| lista zależności | requirements file (`requirements.txt`) | biblioteki potrzebne aplikacji |
| test automatyczny | automated test | program sprawdzający, czy kod działa |
| rozwidlenie repozytorium | fork | własna kopia cudzego repozytorium |
| zatwierdzenie zmian | commit | zapisanie wersji kodu w historii Git |
| wypchnięcie zmian | push | wysłanie commitów na GitHub |
| chmura obliczeniowa | cloud computing | serwery dostępne przez internet |
| dziennik zdarzeń | logs | komunikaty serwera, pomocne przy błędach |